# Train the PathSnap Extra Trees model in Colab

This uses a Linux runtime, avoiding the Windows NumPy binary blocked by Smart App Control. **Running this notebook uploads your code and dataset to Google Colab.** Only continue if that is acceptable.

Before opening this notebook in Colab, create two ZIP files in PowerShell from the repository root: one containing `src`, `configs`, `requirements.txt`, `artifacts`, and `experiments`; and one containing only the sibling `data\train` folder. Do not upload the test set. Do not include `.venv` or `.git` in the code ZIP.

In [ ]:
from google.colab import files
from pathlib import Path
import io, shutil, zipfile

work = Path('/content/pathsnap_work')
repo = work / 'ML_HACKATHON_PES'
data_dir = work / 'data'
shutil.rmtree(work, ignore_errors=True)
repo.mkdir(parents=True)
data_dir.mkdir(parents=True)
uploaded = files.upload()  # Select the code ZIP and the data ZIP

for filename, content in uploaded.items():
    zf = zipfile.ZipFile(io.BytesIO(content))
    members = [n for n in zf.namelist() if n and not n.endswith('/')]
    is_code = any(Path(n).name == 'requirements.txt' for n in members)
    destination = repo if is_code else data_dir
    zf.extractall(destination)

# Normalize either a ZIP containing train/<trip_id>/... or just <trip_id>/... entries.
nested = data_dir / 'data'
if nested.is_dir() and (nested / 'train').is_dir():
    for item in nested.iterdir():
        target = data_dir / item.name
        if target.exists(): shutil.rmtree(target) if target.is_dir() else target.unlink()
        shutil.move(str(item), str(target))
if not (data_dir / 'train').is_dir():
    trip_dirs = [p for p in data_dir.iterdir() if p.is_dir()]
    if trip_dirs and all(any(p.glob('*.track')) for p in trip_dirs):
        train_dir = data_dir / 'train'
        train_dir.mkdir()
        for trip_dir in trip_dirs: shutil.move(str(trip_dir), str(train_dir / trip_dir.name))

assert (repo / 'requirements.txt').is_file(), 'Code ZIP must contain requirements.txt at its root.'
assert (repo / 'src' / 'model' / 'train.py').is_file(), 'Code ZIP must contain src/model/train.py.'
assert (data_dir / 'train').is_dir(), 'Data ZIP must contain train/<trip_id>/... files.'
assert len(list((data_dir / 'train').iterdir())) == 80, 'Expected 80 training trip folders.'
print('Uploaded and unpacked code and 80 training trips. No test data was uploaded.')

In [ ]:
%cd /content/pathsnap_work/ML_HACKATHON_PES
!python -m pip install -r requirements.txt

In [ ]:
# Point the project config at the uploaded data folder, then train on train/ only.
import json
config_path = Path('/content/pathsnap_work/ML_HACKATHON_PES/configs/model.json')
config = json.loads(config_path.read_text())
config['data_root'] = '../data'
config_path.write_text(json.dumps(config, indent=2))
%cd /content/pathsnap_work/ML_HACKATHON_PES
!python -m src.model.train

In [ ]:
# Package model and metrics so they can be extracted into your local repository.
from google.colab import files
import shutil
repo_path = Path('/content/pathsnap_work/ML_HACKATHON_PES')
results_zip = shutil.make_archive('/content/pathsnap_results', 'zip', root_dir=repo_path,
                                   base_dir='artifacts/extra_trees.joblib')
with zipfile.ZipFile(results_zip, 'a', zipfile.ZIP_DEFLATED) as zf:
    zf.write(repo_path / 'artifacts' / 'logistic_regression.joblib', 'artifacts/logistic_regression.joblib')
    zf.write(repo_path / 'experiments' / 'metrics.json', 'experiments/metrics.json')
    zf.write(repo_path / 'experiments' / 'trip_diagnostics.csv', 'experiments/trip_diagnostics.csv')
files.download(results_zip)